## Question

Four hikers must cross a narrow bridge at night. They have one torch, and at most two hikers can cross at a time. Anyone crossing must carry the torch, so it must be brought back whenever people remain on the starting side.

The hikers take **1, 2, 5, and 10 minutes** to cross. A pair moves at the slower person's speed.

**What is the shortest possible time for everyone to get across?**

## Solution

A fastest plan takes **17 minutes**:

| Crossing | Time | Elapsed |
|---|---:|---:|
| 1 and 2 cross | 2 | 2 |
| 1 returns | 1 | 3 |
| 5 and 10 cross | 10 | 13 |
| 2 returns | 2 | 15 |
| 1 and 2 cross | 2 | 17 |

The Python search below checks every legal one- or two-hiker crossing. It treats each combination of hikers' locations and torch position as a state and finds the minimum total crossing time, so it checks plans of every possible length.

## Python


In [1]:
from heapq import heappop, heappush
from itertools import combinations

crossing_times = (1, 2, 5, 10)
hiker_count = len(crossing_times)
all_hikers = (1 << hiker_count) - 1
start = (all_hikers, 0)  # left-side bitmask, torch side (0=left, 1=right)
goal = (0, 1)

shortest_time = {start: 0}
previous = {}
queue = [(0, start)]

while queue:
    elapsed, state = heappop(queue)
    if elapsed != shortest_time[state]:
        continue
    if state == goal:
        break

    left_side, torch_side = state
    available = left_side if torch_side == 0 else all_hikers ^ left_side
    hikers = [i for i in range(hiker_count) if available & (1 << i)]

    for group_size in (1, 2):
        for group in combinations(hikers, group_size):
            group_mask = sum(1 << hiker for hiker in group)
            if torch_side == 0:
                next_left_side = left_side ^ group_mask
            else:
                next_left_side = left_side | group_mask
            next_state = (next_left_side, 1 - torch_side)
            next_elapsed = elapsed + max(crossing_times[i] for i in group)

            if next_elapsed < shortest_time.get(next_state, float("inf")):
                shortest_time[next_state] = next_elapsed
                previous[next_state] = (state, group, next_elapsed - elapsed)
                heappush(queue, (next_elapsed, next_state))

plan = []
state = goal
while state != start:
    prior_state, group, duration = previous[state]
    names = tuple(crossing_times[i] for i in group)
    direction = "right" if state[1] == 1 else "left"
    plan.append((names, direction, duration))
    state = prior_state
plan.reverse()

print(f"Shortest time: {shortest_time[goal]} minutes")
print("Optimal crossing plan:")
for hikers, direction, duration in plan:
    print(f"  {hikers} cross {direction} ({duration} min)")

Shortest time: 17 minutes
Optimal crossing plan:
  (1, 2) cross right (2 min)
  (1,) cross left (1 min)
  (5, 10) cross right (10 min)
  (2,) cross left (2 min)
  (1, 2) cross right (2 min)


## Python interpretation

The shortest-path search finds the 17-minute plan shown above. Because it explores every reachable arrangement of hikers and torch, it also rules out any faster plan with more or fewer crossings.